In [2]:
"""
ORB / Initial Balance structural statistics.
================================================================================

Everything measured on 1-minute bars, US/Eastern.

    ORB   09:30 - 09:44   (15 minutes)
    IB    09:30 - 10:29   (60 minutes)
    RTH   09:30 - 15:59

DEFINITIONS
-----------
  "formed first"    timestamp of the range high vs the range low inside the window
  "close half"      close of the last bar in the window vs the window midpoint
  "breaks first"    first bar trading STRICTLY beyond the range high or low.
                    ORB break measured from 09:45, IB break from 10:30, both to 15:59.
  "held extreme"    the side of the range that did NOT break
  "size quintile"   ORB range / trailing 20-day median ORB range, split into fifths.
                    Q1 = tightest, Q5 = widest.

WHAT IT PRODUCES  (each is a DataFrame in RESULTS, printed/displayed in order)
------------------------------------------------------------------------------
  1  orb_first_extreme          ORB high/low formed first  ->  which side breaks
  2  orb_close_half             ORB close upper/lower half ->  which side breaks
  3  orb_combined               both together
  4  ib_first_extreme           same three questions for the IB
  5  ib_close_half
  6  ib_combined
  7  orb_triple_to_ib           full ORB triple -> IB first extreme / close / break
  8  orb_break_to_ib            ORB break direction alone -> IB first extreme
  9  orb_extreme_survival       does the ORB extreme become the IB extreme
 10  survival_by_break_time     same, split by when the ORB broke
 11  ib_extreme_quarter         which quarter of the hour forms the IB high/low
 12  location_orb_clock         price location in ORB fifths, by 5-min bucket
 13  location_ib_clock          price location in IB fifths, by 5-min bucket
 14  location_since_break       ORB fifths indexed by minutes SINCE the break
 15  retracement_zones          how deep price pulls back before the IB extreme
 16  retracement_by_quintile
 17  invalidation               mid / held-extreme violation rates and timing
 18  survival_curve             held extreme and mid survival at clock cutoffs
 19  targets_conditional        target hit rates given the mid holds N minutes
 20  targets_measured           measured-move targets from the ORB break
 21  targets_structural         prior day / Asia / London / overnight targets
 22  targets_points             fixed point targets, by era
 23  quintile_master            every core statistic, by ORB size quintile
 24  post_ib_by_quintile        10:30-16:00 behaviour, by quintile
 25  post_ib_cont_rev           continuation vs reversal split
 26  ib_projections             +/- k x IB range, touch probability and timing
 27  extension_distribution     percentiles of max extension beyond the IB
 28  extension_by_quintile

METHOD NOTES
------------
  * 1-minute resolution throughout. A bar whose high exceeds one side and whose
    low exceeds the other is recorded as 'both' and excluded from directional
    percentages (it is rare - roughly 1 day in 1500).
  * Half sessions are dropped: only days with a complete 390-bar RTH are used.
  * The 20-day quintile baseline consumes the first 20 days of the sample.
  * Point-denominated results are era-dependent (median ORB was 58 pts in 2020
    and 122 pts in 2026), so those tables are always split by year.

Usage:  edit CONFIG, then  python nq_orb_ib_analysis.py
        or, in a notebook:  RESULTS = run_all()
"""

import os
import numpy as np
import pandas as pd

# ============================================================================
# CONFIG
# ============================================================================

DATA_PATH = r"nq_combined_1min_bars_2020_2026.parquet"
TZ = "US/Eastern"
OUT_DIR = "orb_ib_output"

START_DATE = None          # e.g. "2022-01-01" to restrict the sample
END_DATE = None

RTH_OPEN = 9 * 60 + 30     # 09:30
ORB_END = 9 * 60 + 45      # exclusive
IB_END = 10 * 60 + 30      # exclusive
RTH_CLOSE = 16 * 60        # exclusive (last bar starts 15:59)
FULL_DAY_BARS = 390

QUINTILE_LOOKBACK = 20     # days in the ORB-size baseline
MID_HOLD_WINDOWS = [15, 30]
MEASURED_TARGETS = [0.5, 1.0, 1.5, 2.0]      # x ORB range, from the broken level
POINT_TARGETS = [10, 20, 30, 50]
IB_PROJECTIONS = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]   # x IB range
PERCENTILES = [10, 25, 50, 75, 90, 95]

# Overnight sessions, ET. Asia starts the previous calendar day.
ASIA = (18 * 60, 2 * 60)       # 18:00 prior day -> 02:00
LONDON = (2 * 60, 8 * 60)      # 02:00 -> 08:00

WRITE_CSV = True
WRITE_EXCEL = False
FLOAT_FMT = "{:.1f}"


# ============================================================================
# LOADING
# ============================================================================

def _timestamp_index(df, tz):
    """
    Find the timestamps wherever they live - in a column, or in the index that
    pandas restored from the parquet - and return a DatetimeIndex.

    Everything here operates on DatetimeIndex rather than Series: a parquet
    round-trip often puts the timestamp back as the index, and an index has no
    .dt accessor.
    """
    cols = {str(c).lower(): c for c in df.columns}
    if "timestamp" in cols:
        idx = pd.DatetimeIndex(pd.to_datetime(df[cols["timestamp"]]))
    elif "datetime" in cols:
        idx = pd.DatetimeIndex(pd.to_datetime(df[cols["datetime"]]))
    elif "date" in cols and "time" in cols:
        idx = pd.DatetimeIndex(pd.to_datetime(
            df[cols["date"]].astype(str) + " " + df[cols["time"]].astype(str)))
    else:
        idx = pd.DatetimeIndex(pd.to_datetime(df.index))

    if tz is None:
        return idx
    if idx.tz is not None:                       # already tz-aware -> convert
        return idx.tz_convert(tz)
    try:                                         # naive -> assume exchange local
        return idx.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    except Exception as exc:                     # e.g. tzdata not installed
        print(f"[warn] timezone '{tz}' unavailable ({exc}); "
              f"treating timestamps as already exchange-local.")
        return idx


def load_1min(path=DATA_PATH, tz=TZ, start=START_DATE, end=END_DATE):
    """Load 1-minute bars and return a frame indexed by tz-aware ET timestamps."""
    ext = os.path.splitext(path)[1].lower()
    df = pd.read_parquet(path) if ext in (".parquet", ".pq") else pd.read_csv(path)

    idx = _timestamp_index(df, tz)
    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]
    df.index = idx
    df = df[df.index.notna()].sort_index()

    missing = [c for c in ("open", "high", "low", "close") if c not in df.columns]
    if missing:
        raise KeyError(f"missing OHLC column(s) {missing}; found {list(df.columns)}")

    df = df[["open", "high", "low", "close"]].astype(float)
    if start:
        df = df.loc[start:]
    if end:
        df = df.loc[:end]
    if df.empty:
        raise ValueError("no rows loaded - check DATA_PATH, START_DATE and END_DATE")
    return df


def _prepare(df):
    """Attach session date and minute-of-day; return frame plus a day index."""
    out = df.copy()
    out["mod"] = out.index.hour * 60 + out.index.minute
    out["day"] = out.index.normalize()
    return out


# ============================================================================
# PER-DAY FEATURE TABLE
# ============================================================================

def _window(arr_h, arr_l, mods, lo, hi):
    m = (mods >= lo) & (mods < hi)
    return arr_h[m].max(), arr_l[m].min(), mods[m]


def build_days(df):
    """
    One row per RTH day carrying ORB geometry, IB geometry, break outcomes,
    overnight session levels and prior-day levels.
    """
    d = _prepare(df)
    rth = d[(d["mod"] >= RTH_OPEN) & (d["mod"] < RTH_CLOSE)]
    counts = rth.groupby("day").size()
    good_days = counts[counts == FULL_DAY_BARS].index

    # overnight sessions, keyed by the RTH day they precede
    on = d[(d["mod"] >= ASIA[0]) | (d["mod"] < LONDON[1])].copy()
    on["sess_day"] = np.where(on["mod"] >= ASIA[0],
                              on["day"] + pd.Timedelta(days=1), on["day"])
    asia = on[(on["mod"] >= ASIA[0]) | (on["mod"] < ASIA[1])]
    lon = on[(on["mod"] >= LONDON[0]) & (on["mod"] < LONDON[1])]
    asia_lv = asia.groupby("sess_day").agg(asia_hi=("high", "max"), asia_lo=("low", "min"),
                                           asia_n=("high", "size"))
    lon_lv = lon.groupby("sess_day").agg(lon_hi=("high", "max"), lon_lo=("low", "min"),
                                         lon_n=("high", "size"))

    rows = []
    prev_hi = prev_lo = np.nan
    for day, g in rth.groupby("day"):
        if day not in good_days:
            continue
        h = g["high"].to_numpy()
        l = g["low"].to_numpy()
        c = g["close"].to_numpy()
        o = g["open"].to_numpy()
        mods = g["mod"].to_numpy()

        i_orb = np.flatnonzero(mods < ORB_END)
        i_ib = np.flatnonzero(mods < IB_END)

        ohi, olo = h[i_orb].max(), l[i_orb].min()
        ihi, ilo = h[i_ib].max(), l[i_ib].min()
        orng, irng = ohi - olo, ihi - ilo
        o_mhi = mods[i_orb][np.argmax(h[i_orb])]
        o_mlo = mods[i_orb][np.argmin(l[i_orb])]
        i_mhi = mods[i_ib][np.argmax(h[i_ib])]
        i_mlo = mods[i_ib][np.argmin(l[i_ib])]
        oclose = c[i_orb][-1]
        iclose = c[i_ib][-1]

        def first_break(hi, lo, from_mod):
            m = mods >= from_mod
            up = np.flatnonzero(m & (h > hi))
            dn = np.flatnonzero(m & (l < lo))
            fu = up[0] if len(up) else np.inf
            fd = dn[0] if len(dn) else np.inf
            if fu == np.inf and fd == np.inf:
                return "none", np.nan
            if fu == fd:
                return "both", mods[int(fu)]
            return ("up", mods[int(fu)]) if fu < fd else ("down", mods[int(fd)])

        obrk, obrk_mod = first_break(ohi, olo, ORB_END)
        ibrk, ibrk_mod = first_break(ihi, ilo, IB_END)

        pm = mods >= IB_END
        rows.append(dict(
            day=day, year=day.year,
            o_open=o[0],
            ohi=ohi, olo=olo, orng=orng, omid=(ohi + olo) / 2,
            o_first="high" if o_mhi < o_mlo else ("low" if o_mlo < o_mhi else "same"),
            o_half="upper" if oclose > (ohi + olo) / 2 else ("lower" if oclose < (ohi + olo) / 2 else "mid"),
            obrk=obrk, obrk_mod=obrk_mod,
            ihi=ihi, ilo=ilo, irng=irng, imid=(ihi + ilo) / 2,
            i_first="high" if i_mhi < i_mlo else ("low" if i_mlo < i_mhi else "same"),
            i_half="upper" if iclose > (ihi + ilo) / 2 else ("lower" if iclose < (ihi + ilo) / 2 else "mid"),
            ibrk=ibrk, ibrk_mod=ibrk_mod,
            i_mhi=i_mhi, i_mlo=i_mlo,
            orb_hi_is_ib_hi=bool(ohi == ihi), orb_lo_is_ib_lo=bool(olo == ilo),
            rth_hi=h.max(), rth_lo=l.min(),
            pm_hi=h[pm].max(), pm_lo=l[pm].min(), pm_close=c[-1],
            prev_hi=prev_hi, prev_lo=prev_lo,
        ))
        prev_hi, prev_lo = h.max(), l.min()

    days = pd.DataFrame(rows).set_index("day")
    days = days.join(asia_lv).join(lon_lv)
    days.loc[days["asia_n"].fillna(0) < 300, ["asia_hi", "asia_lo"]] = np.nan
    days.loc[days["lon_n"].fillna(0) < 300, ["lon_hi", "lon_lo"]] = np.nan
    days["prev_mid"] = (days["prev_hi"] + days["prev_lo"]) / 2
    days["on_hi"] = days[["asia_hi", "lon_hi"]].max(axis=1)
    days["on_lo"] = days[["asia_lo", "lon_lo"]].min(axis=1)

    base = days["orng"].shift(1).rolling(QUINTILE_LOOKBACK).median()
    days["rel_size"] = days["orng"] / base
    ok = days["rel_size"].notna()
    days["quintile"] = np.nan
    days.loc[ok, "quintile"] = pd.qcut(days.loc[ok, "rel_size"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
    days["ib_mult"] = days["irng"] / days["orng"]
    return days


def build_paths(df, days):
    """
    Per-day path facts that need the intraday tape: retracement depth before the
    IB extreme, invalidation timing, target hit timing, post-IB extension.
    """
    d = _prepare(df)
    rth = d[(d["mod"] >= RTH_OPEN) & (d["mod"] < RTH_CLOSE)]
    recs = []
    for day, g in rth.groupby("day"):
        if day not in days.index:
            continue
        r = days.loc[day]
        h = g["high"].to_numpy(); l = g["low"].to_numpy(); c = g["close"].to_numpy()
        mods = g["mod"].to_numpy()
        rec = dict(day=day)

        # ---- ORB break leg -------------------------------------------------
        if r["obrk"] in ("up", "down") and r["obrk_mod"] < IB_END:
            up = r["obrk"] == "up"
            brk = r["ohi"] if up else r["olo"]
            held = r["olo"] if up else r["ohi"]
            after = mods > r["obrk_mod"]

            def first_mod(mask):
                idx = np.flatnonzero(after & mask)
                return mods[idx[0]] if len(idx) else np.nan

            rec["mid_loss_mod"] = first_mod(l <= r["omid"] if up else h >= r["omid"])
            rec["held_loss_mod"] = first_mod(l <= held if up else h >= held)

            # deepest pullback before the IB extreme forms
            t_ext = r["i_mhi"] if up else r["i_mlo"]
            if t_ext > r["obrk_mod"]:
                w = after & (mods <= t_ext)
                deep = l[w].min() if up else h[w].max()
                if (deep > r["ohi"]) if up else (deep < r["olo"]):
                    zone = "no retrace"
                elif (deep >= r["omid"]) if up else (deep <= r["omid"]):
                    zone = "near half"
                elif (deep >= r["olo"]) if up else (deep <= r["ohi"]):
                    zone = "far half"
                else:
                    zone = "through held side"
                rec["retrace_zone"] = zone
                rec["move_to_ib_extreme"] = abs((r["ihi"] if up else r["ilo"]) - brk) / r["orng"]

            mfe = (h[after].max() - brk) if up else (brk - l[after].min())
            rec["orb_mfe_pts"] = mfe
            rec["orb_mfe_x"] = mfe / r["orng"]
            for k in MEASURED_TARGETS:
                lvl = brk + k * r["orng"] if up else brk - k * r["orng"]
                rec[f"t_meas_{k}"] = first_mod(h >= lvl if up else l <= lvl)
            for p in POINT_TARGETS:
                lvl = brk + p if up else brk - p
                rec[f"t_pts_{p}"] = first_mod(h >= lvl if up else l <= lvl)
            for name, lvl in (("prev_extreme", r["prev_hi"] if up else r["prev_lo"]),
                              ("prev_mid", r["prev_mid"]),
                              ("asia", r["asia_hi"] if up else r["asia_lo"]),
                              ("london", r["lon_hi"] if up else r["lon_lo"]),
                              ("overnight", r["on_hi"] if up else r["on_lo"])):
                if pd.isna(lvl) or ((lvl <= brk) if up else (lvl >= brk)):
                    rec[f"t_{name}"] = np.nan
                    rec[f"t_{name}_applies"] = False
                else:
                    rec[f"t_{name}"] = first_mod(h >= lvl if up else l <= lvl)
                    rec[f"t_{name}_applies"] = True
                    rec[f"t_{name}_dist"] = abs(lvl - brk) / r["orng"]

        # ---- IB break leg --------------------------------------------------
        if r["ibrk"] in ("up", "down"):
            up = r["ibrk"] == "up"
            brk = r["ihi"] if up else r["ilo"]
            held = r["ilo"] if up else r["ihi"]
            after = mods > r["ibrk_mod"]
            rec["ib_cont"] = (r["obrk"] == r["ibrk"])
            rec["ib_held_holds"] = not bool((l[after] <= held).any() if up else (h[after] >= held).any())
            rec["ib_mid_loss"] = bool((l[after] <= r["imid"]).any() if up else (h[after] >= r["imid"]).any())
            mfe = (h[after].max() - brk) if up else (brk - l[after].min())
            rec["ib_mfe_pts"] = mfe
            rec["ib_mfe_x"] = mfe / r["irng"]
            rec["ib_close_beyond"] = bool(c[-1] > brk) if up else bool(c[-1] < brk)

        # ---- IB-range projections (unconditional, from 10:30) --------------
        pm = mods >= IB_END
        for k in IB_PROJECTIONS:
            up_lvl = r["ihi"] + k * r["irng"]
            dn_lvl = r["ilo"] - k * r["irng"]
            iu = np.flatnonzero(pm & (h >= up_lvl))
            idn = np.flatnonzero(pm & (l <= dn_lvl))
            rec[f"proj_up_{k}"] = mods[iu[0]] if len(iu) else np.nan
            rec[f"proj_dn_{k}"] = mods[idn[0]] if len(idn) else np.nan
        rec["ext_up_x"] = max(0.0, r["pm_hi"] - r["ihi"]) / r["irng"]
        rec["ext_dn_x"] = max(0.0, r["ilo"] - r["pm_lo"]) / r["irng"]
        rec["ext_up_pts"] = max(0.0, r["pm_hi"] - r["ihi"])
        rec["ext_dn_pts"] = max(0.0, r["ilo"] - r["pm_lo"])
        recs.append(rec)

    return pd.DataFrame(recs).set_index("day")


def build_clock(df, days):
    """Price location in range fifths, sampled at every 5-minute bucket close."""
    d = _prepare(df)
    rth = d[(d["mod"] >= RTH_OPEN) & (d["mod"] < RTH_CLOSE)]
    labels = ["below", "F1 0-20%", "F2 20-40%", "F3 40-60%", "F4 60-80%", "F5 80-100%", "above"]
    orb_rows, ib_rows, since_rows = [], [], []
    for day, g in rth.groupby("day"):
        if day not in days.index:
            continue
        r = days.loc[day]
        c = g["close"].to_numpy(); h = g["high"].to_numpy(); l = g["low"].to_numpy()
        mods = g["mod"].to_numpy()
        pos = {m: i for i, m in enumerate(mods)}

        def classify(p, hi, lo):
            if p > hi:
                return labels[6]
            if p < lo:
                return labels[0]
            f = (p - lo) / (hi - lo)
            return labels[1 + min(4, int(f * 5))]

        for start in range(ORB_END, RTH_CLOSE, 5):
            e = min(start + 4, RTH_CLOSE - 1)
            if e in pos:
                orb_rows.append((day, e, classify(c[pos[e]], r["ohi"], r["olo"]), r["obrk"]))
        for start in range(IB_END, RTH_CLOSE, 5):
            e = min(start + 4, RTH_CLOSE - 1)
            if e in pos:
                ib_rows.append((day, e, classify(c[pos[e]], r["ihi"], r["ilo"]), r["ibrk"]))

        # break-oriented: F1 nearest the broken edge, F5 nearest the held edge
        if r["obrk"] in ("up", "down") and r["obrk_mod"] < IB_END:
            up = r["obrk"] == "up"
            for k in range(1, 13):
                m = int(r["obrk_mod"]) + 5 * k
                if m not in pos:
                    continue
                p = c[pos[m]]
                if up:
                    if p > r["ohi"]:
                        z = "beyond"
                    elif p < r["olo"]:
                        z = "opposite side"
                    else:
                        z = f"F{1 + min(4, int(((r['ohi'] - p) / r['orng']) * 5))}"
                else:
                    if p < r["olo"]:
                        z = "beyond"
                    elif p > r["ohi"]:
                        z = "opposite side"
                    else:
                        z = f"F{1 + min(4, int(((p - r['olo']) / r['orng']) * 5))}"
                since_rows.append((day, 5 * k, z, r["obrk"]))

    orb = pd.DataFrame(orb_rows, columns=["day", "mod", "zone", "obrk"])
    ib = pd.DataFrame(ib_rows, columns=["day", "mod", "zone", "ibrk"])
    since = pd.DataFrame(since_rows, columns=["day", "mins_since_break", "zone", "obrk"])
    return orb, ib, since


# ============================================================================
# HELPERS
# ============================================================================

def _hhmm(m):
    return f"{int(m) // 60:02d}:{int(m) % 60:02d}"


def _dir_table(frame, by, brk_col, label):
    """Percent of up/down first breaks, grouped by `by`."""
    f = frame[frame[brk_col].isin(["up", "down"])]
    g = f.groupby(by)[brk_col].value_counts().unstack(fill_value=0)
    for c in ("up", "down"):
        if c not in g:
            g[c] = 0
    out = pd.DataFrame({
        "n": g["up"] + g["down"],
        "up first %": 100 * g["up"] / (g["up"] + g["down"]),
        "down first %": 100 * g["down"] / (g["up"] + g["down"]),
    })
    nb = frame[~frame[brk_col].isin(["up", "down"])].groupby(by).size()
    out["no break"] = nb.reindex(out.index).fillna(0).astype(int)
    out.index.name = label
    return out.round(1)


def _pct(series):
    return round(100 * float(series.mean()), 1)


# ============================================================================
# SECTIONS
# ============================================================================

def s1_orb_structure(days):
    d = days[(days["o_first"] != "same") & (days["o_half"] != "mid")]
    t1 = _dir_table(d, d["o_first"] + " formed first", "obrk", "ORB structure")
    t2 = _dir_table(d, "close " + d["o_half"] + " half", "obrk", "ORB structure")
    t3 = _dir_table(d, d["o_first"] + " first + close " + d["o_half"], "obrk", "ORB structure")
    return t1, t2, t3


def s2_ib_structure(days):
    d = days[(days["i_first"] != "same") & (days["i_half"] != "mid")]
    t1 = _dir_table(d, d["i_first"] + " formed first", "ibrk", "IB structure")
    t2 = _dir_table(d, "close " + d["i_half"] + " half", "ibrk", "IB structure")
    t3 = _dir_table(d, d["i_first"] + " first + close " + d["i_half"], "ibrk", "IB structure")
    return t1, t2, t3


def s3_orb_to_ib(days):
    d = days[(days["o_first"] != "same") & (days["o_half"] != "mid") &
             days["obrk"].isin(["up", "down"])]
    key = ("ORB " + d["o_first"] + " first / close " + d["o_half"] + " / " + d["obrk"] + " break")
    rows = []
    for k, g in d.groupby(key):
        gb = g[g["ibrk"].isin(["up", "down"])]
        rows.append(dict(cell=k, n=len(g),
                         **{"IB high first %": _pct(g["i_first"] == "high"),
                            "IB close upper %": _pct(g["i_half"] == "upper"),
                            "IB breaks up %": _pct(gb["ibrk"] == "up"),
                            "no break": int((g["ibrk"] == "none").sum())}))
    triple = pd.DataFrame(rows).set_index("cell").sort_index()

    b = days[days["obrk"].isin(["up", "down"])]
    rows = []
    for k, g in b.groupby("obrk"):
        gb = g[g["ibrk"].isin(["up", "down"])]
        rows.append(dict(cell=f"ORB broke {k}", n=len(g),
                         **{"IB high first %": _pct(g["i_first"] == "high"),
                            "IB low first %": _pct(g["i_first"] == "low"),
                            "IB breaks up %": _pct(gb["ibrk"] == "up")}))
    by_break = pd.DataFrame(rows).set_index("cell")
    return triple, by_break


def s3b_survival(days):
    n = len(days)
    overall = pd.DataFrame([dict(
        cell="all days", n=n,
        **{"ORB high = IB high %": _pct(days["orb_hi_is_ib_hi"]),
           "ORB low = IB low %": _pct(days["orb_lo_is_ib_lo"]),
           "both %": _pct(days["orb_hi_is_ib_hi"] & days["orb_lo_is_ib_lo"]),
           "neither %": _pct(~days["orb_hi_is_ib_hi"] & ~days["orb_lo_is_ib_lo"])})]).set_index("cell")
    b = days[days["obrk"].isin(["up", "down"])].copy()
    b["held_holds"] = np.where(b["obrk"] == "up", b["orb_lo_is_ib_lo"], b["orb_hi_is_ib_hi"])
    rows = []
    for k, g in b.groupby("obrk"):
        rows.append(dict(cell=f"ORB broke {k}", n=len(g),
                         **{"ORB high = IB high %": _pct(g["orb_hi_is_ib_hi"]),
                            "ORB low = IB low %": _pct(g["orb_lo_is_ib_lo"]),
                            "held side holds %": _pct(g["held_holds"])}))
    by_break = pd.DataFrame(rows).set_index("cell")

    bins = [(ORB_END, ORB_END + 5, "09:45-09:49"), (ORB_END + 5, ORB_END + 15, "09:50-09:59"),
            (ORB_END + 15, ORB_END + 30, "10:00-10:14"), (ORB_END + 30, IB_END, "10:15-10:29"),
            (IB_END, RTH_CLOSE, "after 10:30")]
    rows = []
    for lo, hi, lab in bins:
        for dirn in ("up", "down"):
            g = b[(b["obrk"] == dirn) & (b["obrk_mod"] >= lo) & (b["obrk_mod"] < hi)]
            if len(g) < 5:
                continue
            rows.append(dict(break_window=lab, direction=dirn, n=len(g),
                             **{"IB high first %": _pct(g["i_first"] == "high"),
                                "held side holds %": _pct(g["held_holds"])}))
    by_time = pd.DataFrame(rows).set_index(["break_window", "direction"])
    return overall, by_break, by_time


def s4_quarter(days):
    q = lambda mod: (np.asarray(mod) - RTH_OPEN) // 15
    labels = ["Q1 09:30-09:44", "Q2 09:45-09:59", "Q3 10:00-10:14", "Q4 10:15-10:29"]
    hi = pd.Series(q(days["i_mhi"])).value_counts(normalize=True).sort_index()
    lo = pd.Series(q(days["i_mlo"])).value_counts(normalize=True).sort_index()
    out = pd.DataFrame({"IB high %": 100 * hi, "IB low %": 100 * lo})
    out.index = [labels[i] for i in out.index]
    out.index.name = "quarter"

    b = days[days["obrk"].isin(["up", "down"])]
    rows = []
    for k, g in b.groupby("obrk"):
        h = pd.Series(q(g["i_mhi"])).value_counts(normalize=True).sort_index()
        l = pd.Series(q(g["i_mlo"])).value_counts(normalize=True).sort_index()
        for i, lab in enumerate(labels):
            rows.append(dict(direction=f"ORB broke {k}", quarter=lab,
                             **{"IB high %": round(100 * h.get(i, 0), 1),
                                "IB low %": round(100 * l.get(i, 0), 1)}))
    split = pd.DataFrame(rows).set_index(["direction", "quarter"])
    return out.round(1), split


def s5_location(orb_clock, ib_clock, since):
    def pivot(frame, idx, cols="zone"):
        p = pd.crosstab(frame[idx], frame[cols], normalize="index") * 100
        order = [c for c in ["below", "F1 0-20%", "F2 20-40%", "F3 40-60%", "F4 60-80%",
                             "F5 80-100%", "above", "beyond", "F1", "F2", "F3", "F4", "F5",
                             "opposite side"] if c in p.columns]
        p = p[order]
        inside = [c for c in p.columns if c.startswith("F")]
        p["inside %"] = p[inside].sum(axis=1)
        return p.round(1)

    a = pivot(orb_clock, "mod"); a.index = [_hhmm(m) for m in a.index]; a.index.name = "bucket close"
    b = pivot(ib_clock, "mod"); b.index = [_hhmm(m) for m in b.index]; b.index.name = "bucket close"
    c = pivot(since, "mins_since_break"); c.index.name = "minutes since ORB break"
    return a, b, c


def s6_retracement(days, paths):
    j = days.join(paths[["retrace_zone", "move_to_ib_extreme"]])
    j = j[j["retrace_zone"].notna()]
    rows = []
    for dirn in ("up", "down"):
        g = j[j["obrk"] == dirn]
        vc = g["retrace_zone"].value_counts(normalize=True) * 100
        for z in ["no retrace", "near half", "far half", "through held side"]:
            rows.append(dict(direction=f"ORB broke {dirn}", zone=z,
                             pct=round(vc.get(z, 0.0), 1),
                             n=int((g["retrace_zone"] == z).sum()),
                             median_move_x_orb=round(float(
                                 g.loc[g["retrace_zone"] == z, "move_to_ib_extreme"].median()), 2)))
    overall = pd.DataFrame(rows).set_index(["direction", "zone"])

    rows = []
    for q, g in j.groupby("quintile"):
        vc = g["retrace_zone"].value_counts(normalize=True) * 100
        rows.append(dict(quintile=f"Q{int(q)}", n=len(g),
                         **{z: round(vc.get(z, 0.0), 1) for z in
                            ["no retrace", "near half", "far half", "through held side"]},
                         median_move_x_orb=round(float(g["move_to_ib_extreme"].median()), 2)))
    by_q = pd.DataFrame(rows).set_index("quintile")
    return overall, by_q


def s7_invalidation(days, paths):
    j = days.join(paths[["mid_loss_mod", "held_loss_mod"]])
    j = j[j["obrk"].isin(["up", "down"]) & (j["obrk_mod"] < IB_END)]
    n = len(j)
    rows = [
        dict(event="trades back through the ORB mid",
             pct=round(100 * j["mid_loss_mod"].notna().mean(), 1),
             median_min_after_break=int((j["mid_loss_mod"] - j["obrk_mod"]).median())),
        dict(event="trades through the held extreme",
             pct=round(100 * j["held_loss_mod"].notna().mean(), 1),
             median_min_after_break=int((j["held_loss_mod"] - j["obrk_mod"]).median())),
    ]
    lost = j[j["mid_loss_mod"].notna()]
    rows.append(dict(event="of days losing the mid, also lose the held extreme",
                     pct=round(100 * lost["held_loss_mod"].notna().mean(), 1),
                     median_min_after_break=np.nan))
    summary = pd.DataFrame(rows).set_index("event")

    cuts = [(IB_END, "10:30 (IB end)"), (11 * 60, "11:00"), (12 * 60, "12:00"),
            (13 * 60, "13:00"), (15 * 60, "15:00"), (RTH_CLOSE, "15:59")]
    rows = []
    for cut, lab in cuts:
        rows.append(dict(by=lab,
                         **{"held extreme violated %": round(100 * (j["held_loss_mod"] < cut).mean(), 1),
                            "held extreme survives %": round(100 * (1 - (j["held_loss_mod"] < cut).mean()), 1),
                            "ORB mid violated %": round(100 * (j["mid_loss_mod"] < cut).mean(), 1)}))
    curve = pd.DataFrame(rows).set_index("by")

    rows = []
    conds = [("all break days", pd.Series(True, index=j.index))]
    for w in MID_HOLD_WINDOWS:
        conds.append((f"mid holds {w} min after break",
                      j["mid_loss_mod"].isna() | (j["mid_loss_mod"] > j["obrk_mod"] + w)))
    conds.append(("mid lost within 15 min", j["mid_loss_mod"] <= j["obrk_mod"] + 15))
    p = paths.reindex(j.index)
    for lab, mask in conds:
        g = j[mask]
        gp = p[mask]
        row = dict(condition=lab, n=int(mask.sum()),
                   share_of_days=round(100 * mask.mean(), 1))
        for k in MEASURED_TARGETS:
            row[f"{k}x hit %"] = round(100 * gp[f"t_meas_{k}"].notna().mean(), 1)
        row["loses held extreme %"] = round(100 * g["held_loss_mod"].notna().mean(), 1)
        rows.append(row)
    conditional = pd.DataFrame(rows).set_index("condition")
    return summary, curve, conditional


def s8_targets(days, paths):
    j = days.join(paths, rsuffix="_p")
    j = j[j["obrk"].isin(["up", "down"]) & (j["obrk_mod"] < IB_END)]
    n = len(j)
    rows = []
    for k in MEASURED_TARGETS:
        t = j[f"t_meas_{k}"]
        clean = t.notna() & (j["mid_loss_mod"].isna() | (t < j["mid_loss_mod"]))
        rows.append(dict(target=f"measured {k}x ORB", applicable_pct=100.0,
                         hit_by_1600=round(100 * t.notna().mean(), 1),
                         hit_before_mid_loss=round(100 * clean.mean(), 1),
                         median_min=int((t - j["obrk_mod"]).median())))
    measured = pd.DataFrame(rows).set_index("target")

    rows = []
    for name, lab in (("prev_extreme", "prior day high/low"), ("prev_mid", "prior day mid"),
                      ("asia", "Asia high/low"), ("london", "London high/low"),
                      ("overnight", "overnight high/low")):
        ap = j[f"t_{name}_applies"].fillna(False)
        if ap.sum() == 0:
            continue
        g = j[ap]
        t = g[f"t_{name}"]
        clean = t.notna() & (g["mid_loss_mod"].isna() | (t < g["mid_loss_mod"]))
        rows.append(dict(target=lab,
                         applicable_pct=round(100 * ap.mean(), 1),
                         hit_by_1600=round(100 * t.notna().mean(), 1),
                         hit_before_mid_loss=round(100 * clean.mean(), 1),
                         median_dist_x_orb=round(float(g[f"t_{name}_dist"].median()), 2),
                         median_min=int((t - g["obrk_mod"]).median())))
    structural = pd.DataFrame(rows).set_index("target")

    rows = []
    for yr, g in j.groupby("year"):
        row = dict(year=yr, n=len(g), median_orb_pts=round(float(g["orng"].median()), 1),
                   median_mfe_pts=round(float(g["orb_mfe_pts"].median()), 1))
        for p in POINT_TARGETS:
            row[f"{p}pt hit %"] = round(100 * g[f"t_pts_{p}"].notna().mean(), 1)
        rows.append(row)
    points = pd.DataFrame(rows).set_index("year")
    return measured, structural, points


def s9_quintiles(days, paths):
    j = days.join(paths, rsuffix="_p")
    j = j[j["quintile"].notna()]
    out = {}
    for q, g in j.groupby("quintile"):
        b = g[g["obrk"].isin(["up", "down"])]
        held = np.where(b["obrk"] == "up", b["orb_lo_is_ib_lo"], b["orb_hi_is_ib_hi"])
        gb = g[g["ibrk"].isin(["up", "down"])]
        gz = g[g["retrace_zone"].notna()]
        col = {
            "days": len(g),
            "ORB / 20d median": round(float(g["rel_size"].median()), 2),
            "IB / ORB multiple": round(float(g["ib_mult"].median()), 2),
            "close lower -> breaks down %": _pct(b.loc[b["o_half"] == "lower", "obrk"] == "down"),
            "close upper -> breaks up %": _pct(b.loc[b["o_half"] == "upper", "obrk"] == "up"),
            "high first -> breaks down %": _pct(b.loc[b["o_first"] == "high", "obrk"] == "down"),
            "ORB up -> IB low first %": _pct(b.loc[b["obrk"] == "up", "i_first"] == "low"),
            "ORB down -> IB high first %": _pct(b.loc[b["obrk"] == "down", "i_first"] == "high"),
            "held side holds to 10:29 %": round(100 * float(np.mean(held)), 1),
            "IB close lower -> down %": _pct(gb.loc[gb["i_half"] == "lower", "ibrk"] == "down"),
            "IB close upper -> up %": _pct(gb.loc[gb["i_half"] == "upper", "ibrk"] == "up"),
            "IB never breaks %": _pct(g["ibrk"] == "none"),
            "retrace: none %": _pct(gz["retrace_zone"] == "no retrace"),
            "retrace: near half %": _pct(gz["retrace_zone"] == "near half"),
            "retrace: far half %": _pct(gz["retrace_zone"] == "far half"),
            "retrace: through held %": _pct(gz["retrace_zone"] == "through held side"),
            "move break->IB extreme (xORB)": round(float(g["move_to_ib_extreme"].median()), 2),
        }
        out[f"Q{int(q)}"] = col
    return pd.DataFrame(out)


def s10_post_ib(days, paths):
    j = days.join(paths, rsuffix="_p")
    j = j[j["quintile"].notna()]
    out, split = {}, {}
    for q, g in j.groupby("quintile"):
        b = g[g["ibrk"].isin(["up", "down"])]
        out[f"Q{int(q)}"] = {
            "days": len(g),
            "IB breaks %": _pct(g["ibrk"].isin(["up", "down"])),
            "median IB break time": _hhmm(float(b["ibrk_mod"].median())),
            "IB break is continuation %": _pct(b["ib_cont"]),
            "held IB side holds to 16:00 %": _pct(b["ib_held_holds"]),
            "IB mid lost after break %": _pct(b["ib_mid_loss"]),
            "closes beyond broken IB level %": _pct(b["ib_close_beyond"]),
            "reaches 0.5x IB beyond break %": _pct(b["ib_mfe_x"] >= 0.5),
            "reaches 1.0x IB beyond break %": _pct(b["ib_mfe_x"] >= 1.0),
            "median extension (x IB)": round(float(b["ib_mfe_x"].median()), 2),
            "median extension (x ORB)": round(float((b["ib_mfe_pts"] / b["orng"]).median()), 2),
            "RTH high forms after 10:30 %": _pct(g["pm_hi"] >= g["rth_hi"]),
            "RTH low forms after 10:30 %": _pct(g["pm_lo"] <= g["rth_lo"]),
            "post-10:30 range / IB": round(float(((g["pm_hi"] - g["pm_lo"]) / g["irng"]).median()), 2),
        }
        cont, rev = b[b["ib_cont"]], b[~b["ib_cont"].astype(bool)]
        split[f"Q{int(q)}"] = {
            "CONT: held side holds %": _pct(cont["ib_held_holds"]),
            "CONT: median extension (x IB)": round(float(cont["ib_mfe_x"].median()), 2),
            "CONT: reaches 1.0x IB %": _pct(cont["ib_mfe_x"] >= 1.0),
            "REV: held side holds %": _pct(rev["ib_held_holds"]),
            "REV: median extension (x IB)": round(float(rev["ib_mfe_x"].median()), 2),
            "REV: reaches 1.0x IB %": _pct(rev["ib_mfe_x"] >= 1.0),
        }
    return pd.DataFrame(out), pd.DataFrame(split)


def s11_projections(days, paths):
    j = days.join(paths, rsuffix="_p")
    n = len(j)
    first_up = j["ibrk"] == "up"
    first_dn = j["ibrk"] == "down"
    rows = []
    for k in IB_PROJECTIONS:
        for side, lab, cond in (("up", f"IB high + {k:.2f}x IB", first_up),
                                ("dn", f"IB low - {k:.2f}x IB", first_dn)):
            t = j[f"proj_{side}_{k}"]
            rows.append(dict(level=lab,
                             touch_any_day=round(100 * t.notna().mean(), 1),
                             touch_if_broke_that_side=round(100 * t[cond].notna().mean(), 1),
                             median_min_after_1030=int((t - IB_END).median()) if t.notna().any() else np.nan))
    return pd.DataFrame(rows).set_index("level")


def s12_extension(days, paths):
    j = days.join(paths, rsuffix="_p")
    recent = j[j["year"] >= j["year"].max() - 1]
    rows = []
    specs = [("above IB high - all days", j["ext_up_x"], recent["ext_up_pts"]),
             ("below IB low - all days", j["ext_dn_x"], recent["ext_dn_pts"]),
             ("above IB high - IB broke up first", j.loc[j["ibrk"] == "up", "ext_up_x"],
              recent.loc[recent["ibrk"] == "up", "ext_up_pts"]),
             ("below IB low - IB broke down first", j.loc[j["ibrk"] == "down", "ext_dn_x"],
              recent.loc[recent["ibrk"] == "down", "ext_dn_pts"])]
    for lab, v, pts in specs:
        row = dict(series=lab, n=len(v), mean=round(float(v.mean()), 2),
                   sd=round(float(v.std()), 2))
        for p in PERCENTILES:
            row[f"p{p} (xIB)"] = round(float(np.percentile(v, p)), 2)
        for p in PERCENTILES:
            row[f"p{p} (pts, recent)"] = round(float(np.percentile(pts, p)), 0)
        rows.append(row)
    dist = pd.DataFrame(rows).set_index("series")

    rows = []
    for q, g in j[j["quintile"].notna()].groupby("quintile"):
        u = g.loc[g["ibrk"] == "up", "ext_up_x"]
        d = g.loc[g["ibrk"] == "down", "ext_dn_x"]
        rows.append(dict(quintile=f"Q{int(q)}",
                         median_up=round(float(u.median()), 2), p75_up=round(float(u.quantile(.75)), 2),
                         median_dn=round(float(d.median()), 2), p75_dn=round(float(d.quantile(.75)), 2)))
    by_q = pd.DataFrame(rows).set_index("quintile")
    return dist, by_q


# ============================================================================
# RUNNER
# ============================================================================

def _show(name, table, title):
    try:
        from IPython.display import display
        print(f"\n{'=' * 78}\n{name}  -  {title}\n{'=' * 78}")
        display(table)
    except Exception:
        print(f"\n{'=' * 78}\n{name}  -  {title}\n{'=' * 78}")
        with pd.option_context("display.width", 200, "display.max_columns", 60):
            print(table.to_string())


def run_all(path=DATA_PATH, write=WRITE_CSV):
    df = load_1min(path)
    days = build_days(df)
    paths = build_paths(df, days)
    orb_clock, ib_clock, since = build_clock(df, days)

    R = {}
    R["orb_first_extreme"], R["orb_close_half"], R["orb_combined"] = s1_orb_structure(days)
    R["ib_first_extreme"], R["ib_close_half"], R["ib_combined"] = s2_ib_structure(days)
    R["orb_triple_to_ib"], R["orb_break_to_ib"] = s3_orb_to_ib(days)
    R["orb_extreme_survival"], R["survival_by_break"], R["survival_by_break_time"] = s3b_survival(days)
    R["ib_extreme_quarter"], R["ib_extreme_quarter_split"] = s4_quarter(days)
    R["location_orb_clock"], R["location_ib_clock"], R["location_since_break"] = s5_location(orb_clock, ib_clock, since)
    R["retracement_zones"], R["retracement_by_quintile"] = s6_retracement(days, paths)
    R["invalidation"], R["survival_curve"], R["targets_conditional"] = s7_invalidation(days, paths)
    R["targets_measured"], R["targets_structural"], R["targets_points"] = s8_targets(days, paths)
    R["quintile_master"] = s9_quintiles(days, paths)
    R["post_ib_by_quintile"], R["post_ib_cont_rev"] = s10_post_ib(days, paths)
    R["ib_projections"] = s11_projections(days, paths)
    R["extension_distribution"], R["extension_by_quintile"] = s12_extension(days, paths)

    titles = {
        "orb_first_extreme": "which ORB extreme formed first -> which side breaks",
        "orb_close_half": "where the 09:44 close sits -> which side breaks",
        "orb_combined": "first extreme + close half -> which side breaks",
        "ib_first_extreme": "which IB extreme formed first -> which side breaks",
        "ib_close_half": "where the 10:29 close sits -> which side breaks",
        "ib_combined": "first extreme + close half -> which side breaks",
        "orb_triple_to_ib": "full ORB triple -> IB structure",
        "orb_break_to_ib": "ORB break direction -> IB first extreme",
        "orb_extreme_survival": "does the ORB extreme survive as the IB extreme",
        "survival_by_break": "survival by ORB break direction",
        "survival_by_break_time": "survival by when the ORB broke (late breaks are partly circular)",
        "ib_extreme_quarter": "which quarter of the hour forms the IB extreme",
        "ib_extreme_quarter_split": "same, split by ORB break direction",
        "location_orb_clock": "price location in ORB fifths, by 5-min bucket close",
        "location_ib_clock": "price location in IB fifths, by 5-min bucket close",
        "location_since_break": "ORB fifths oriented to the break (F1 = nearest broken edge)",
        "retracement_zones": "deepest pullback before the IB extreme forms",
        "retracement_by_quintile": "same, by ORB size quintile",
        "invalidation": "bias invalidation rates and timing",
        "survival_curve": "held extreme / mid survival at clock cutoffs",
        "targets_conditional": "target hit rates conditional on the mid holding",
        "targets_measured": "measured-move targets from the ORB break",
        "targets_structural": "prior-day and overnight targets",
        "targets_points": "fixed point targets, by year",
        "quintile_master": "every core statistic by ORB size quintile",
        "post_ib_by_quintile": "10:30-16:00 behaviour by quintile",
        "post_ib_cont_rev": "continuation vs reversal split",
        "ib_projections": "+/- k x IB range, touch probability",
        "extension_distribution": "max extension beyond the IB - percentiles",
        "extension_by_quintile": "extension in the break direction by quintile",
    }
    for k, t in R.items():
        _show(k, t, titles.get(k, ""))

    if write:
        os.makedirs(OUT_DIR, exist_ok=True)
        for k, t in R.items():
            t.to_csv(os.path.join(OUT_DIR, f"{k}.csv"))
        if WRITE_EXCEL:
            with pd.ExcelWriter(os.path.join(OUT_DIR, "orb_ib_stats.xlsx")) as xl:
                for k, t in R.items():
                    t.to_excel(xl, sheet_name=k[:31])
        print(f"\nWrote {len(R)} tables to {OUT_DIR}/")

    R["_days"] = days
    R["_paths"] = paths
    return R


if __name__ == "__main__":
    RESULTS = run_all()


orb_first_extreme  -  which ORB extreme formed first -> which side breaks


,n,up first %,down first %,no break
ORB structure,,,,
high formed first,770,30.8,69.2,1
low formed first,794,72.8,27.2,0



orb_close_half  -  where the 09:44 close sits -> which side breaks


,n,up first %,down first %,no break
ORB structure,,,,
close lower half,738,22.4,77.6,1
close upper half,826,78.7,21.3,0



orb_combined  -  first extreme + close half -> which side breaks


,n,up first %,down first %,no break
ORB structure,,,,
high first + close lower,605,20.8,79.2,1
high first + close upper,165,67.3,32.7,0
low first + close lower,133,29.3,70.7,0
low first + close upper,661,81.5,18.5,0


... (51 outputs left out)

,n,mean,sd,p10 (xIB),p25 (xIB),p50 (xIB),p75 (xIB),p90 (xIB),p95 (xIB),"p10 (pts, recent)","p25 (pts, recent)","p50 (pts, recent)","p75 (pts, recent)","p90 (pts, recent)","p95 (pts, recent)"
series,,,,,,,,,,,,,,,
above IB high - all days,1570,0.42,0.60,0.0,0.00,0.20,0.65,1.11,1.50,0.0,0.0,30.0,88.0,161.0,225.0
below IB low - all days,1570,0.46,0.78,0.0,0.00,0.11,0.67,1.34,1.84,0.0,0.0,14.0,115.0,213.0,288.0
above IB high - IB broke up first,810,0.67,0.56,0.1,0.26,0.55,0.90,1.38,1.75,17.0,40.0,72.0,123.0,208.0,264.0
below IB low - IB broke down first,717,0.80,0.83,0.1,0.23,0.58,1.07,1.74,2.27,16.0,38.0,101.0,192.0,275.0,354.0



extension_by_quintile  -  extension in the break direction by quintile


,median_up,p75_up,median_dn,p75_dn
quintile,,,,
Q1,0.63,0.95,0.57,1.21
Q2,0.65,1.05,0.67,1.21
Q3,0.55,0.89,0.62,1.09
Q4,0.46,0.80,0.56,1.04
Q5,0.48,0.83,0.52,0.88



Wrote 30 tables to orb_ib_output/
